In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from ragas.llms import llm_factory
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset
import pandas as pd

# TODO
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    AnswerRelevancy,
    Faithfulness
)
from ragas import evaluate

load_dotenv()

client = OpenAI(
    api_key=os.getenv('API_KEY'),
    base_url="https://foundation-models.api.cloud.ru/v1"
)
llm = llm_factory("Qwen/Qwen3-235B-A22B-Instruct-2507", provider="openai", client=client, max_tokens=8192)
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B"
)

df = pd.read_parquet('../../data/ragas/gd_filled.parquet')
dataset = Dataset.from_pandas(df)

# TODO
result = evaluate(
    dataset=dataset,
    metrics=[
        ContextPrecision(llm=llm),
        ContextRecall(llm=llm),
        AnswerRelevancy(llm=llm, embeddings=embeddings),
        Faithfulness(llm=llm)
    ]
)

print(result)